In [1]:
from __future__ import annotations

import json
import logging
import os
import smtplib
from dataclasses import dataclass
from datetime import datetime, timedelta, timezone
from email.mime.multipart import MIMEMultipart
from email.mime.text import MIMEText
from typing import Any, Dict, List, Optional, Protocol, Sequence

In [7]:
from dotenv import load_dotenv
load_dotenv()

True

In [2]:
@dataclass(frozen=True)
class EmailPayload:
    to: str
    subject: str
    html: str

@dataclass
class AsyncTask:
    task_id: str
    payload: Dict[str, Any]
    retry_count: int
    max_retries: int


In [8]:
class AsyncTaskService(Protocol):
    def get_pending_tasks_for_processing(self, limit: int) -> List[AsyncTask]: ...
    def mark_as_processing(self, task_id: str) -> None: ...
    def mark_as_completed(self, task_id: str) -> None: ...
    def mark_as_failed(self, task_id: str, retry_count: int) -> None: ...
    def schedule_retry(self, task_id: str, retry_count: int, next_retry_at: datetime) -> None: ...


In [60]:
class EmailWorkerService:
    """
    Python equivalent of your NestJS EmailWorkerService.
    - No cron included (call process_email_queue() from your runner)
    - SMTP via Gmail (STARTTLS on 587) using .env
    - Retry intervals: 1m, 5m, 15m, 1h, 4h
    """

    def __init__(
        self,
        async_task_service: AsyncTaskService,
        logger: Optional[logging.Logger] = None,
    ) -> None:
        self.async_task_service = async_task_service
        self.logger = logger or logging.getLogger(self.__class__.__name__)
        self.is_processing = False

        # Config from .env
        self.mail_from = os.getenv("MAIL_FROM") or '"CMS Notifications" <no-reply@cms.local>'
        self.smtp_host = os.getenv("SMTP_HOST") or "localhost"
        self.smtp_port = int(os.getenv("SMTP_PORT") or "587")
        self.smtp_user = os.getenv("SMTP_USER") or None
        self.smtp_pass = os.getenv("SMTP_PASS") or None
        self.smtp_secure = (os.getenv("SMTP_SECURE") or "false").lower() == "true"

        if not os.getenv("SMTP_HOST"):
            self.logger.warning("SMTP_HOST not configured - emails may fail to send.")
        self.logger.info("Email Worker initialized - SMTP: %s:%s", self.smtp_host, self.smtp_port)

    # ----------------------------
    # Main queue processing (NO CRON)
    # ----------------------------
    def process_email_queue(self, limit: int = 10) -> None:
        if self.is_processing:
            return

        self.is_processing = True
        try:
            tasks = self.async_task_service.get_pending_tasks_for_processing(limit)
            if tasks:
                self.logger.info("Processing %d email tasks", len(tasks))
                for task in tasks:
                    self._process_task(task)
        except Exception as e:
            self.logger.exception("Email queue processing error: %s", str(e))
        finally:
            self.is_processing = False


    def run_once(self, batch_size: int = 10):
        """
        Process pending email tasks ONCE (no cron).
        Ideal for Spark / Jupyter jobs.
        """
        tasks = self.async_task_service.get_pending_tasks_for_processing(batch_size)

        if not tasks:
            self.logger.info("No pending email tasks to process")
            return

        self.logger.info(f"Processing {len(tasks)} email tasks")

        for task in tasks:
            self._process_task(task)
       

    # ----------------------------
    # Process a single task
    # ----------------------------
    def _process_task(self, task: AsyncTask) -> None:
        try:
            self.async_task_service.mark_as_processing(task.task_id)

            payload = self._parse_email_payload(task.payload)
            self.logger.debug("Sending email task_id=%s to=%s", task.task_id, payload.to)

            self._send_email(payload)

            self.async_task_service.mark_as_completed(task.task_id)
            self.logger.info("Email sent: %s to %s - %s", task.task_id, payload.to, payload.subject)

        except Exception as e:
            self.logger.exception("Email send failed for %s: %s", task.task_id, str(e))
            self._handle_task_failure(task, e)

    # ----------------------------
    # Failure handling + retry
    # ----------------------------
    def _handle_task_failure(self, task: AsyncTask, error: Exception) -> None:
        new_retry_count = int(task.retry_count) + 1
        error_message = str(error) or "Unknown error"

        if new_retry_count >= int(task.max_retries):
            self.async_task_service.mark_as_failed(task.task_id, new_retry_count)
            self.logger.error(
                "Email task failed permanently: %s after %d attempts - %s",
                task.task_id, new_retry_count, error_message
            )
            return

        next_retry = self._calculate_next_retry(new_retry_count)
        self.async_task_service.schedule_retry(task.task_id, new_retry_count, next_retry)

        self.logger.warning(
            "Retry scheduled: %s (attempt %d/%d) at %s - %s",
            task.task_id, new_retry_count, int(task.max_retries), next_retry.isoformat(), error_message
        )

    @staticmethod
    def _calculate_next_retry(retry_count: int) -> datetime:
        intervals = [60, 300, 900, 3600, 14400]  # 1m, 5m, 15m, 1h, 4h
        delay = intervals[min(retry_count - 1, len(intervals) - 1)]
        return datetime.now(timezone.utc) + timedelta(seconds=delay)

    # ----------------------------
    # SMTP send (Gmail-friendly)
    # ----------------------------
    def _send_email(self, payload: EmailPayload) -> None:
        msg = MIMEMultipart("alternative")
        msg["From"] = self.mail_from
        msg["To"] = payload.to
        msg["Subject"] = payload.subject
        msg.attach(MIMEText(payload.html, "html", "utf-8"))

        # SMTP_SECURE=true would mean implicit SSL (port 465 typically)
        # Your config is SMTP_SECURE=false, so we use STARTTLS on 587.
        if self.smtp_secure:
            # If you ever switch to 465/SSL:
            with smtplib.SMTP_SSL(self.smtp_host, self.smtp_port, timeout=30) as server:
                if self.smtp_user:
                    if not self.smtp_pass:
                        raise RuntimeError("SMTP_USER set but SMTP_PASS missing.")
                    server.login(self.smtp_user, self.smtp_pass)
                server.sendmail(self.mail_from, [payload.to], msg.as_string())
        else:
            with smtplib.SMTP(self.smtp_host, self.smtp_port, timeout=30) as server:
                server.ehlo()
                server.starttls()
                server.ehlo()
                if self.smtp_user:
                    if not self.smtp_pass:
                        raise RuntimeError("SMTP_USER set but SMTP_PASS missing.")
                    server.login(self.smtp_user, self.smtp_pass)
                server.sendmail(self.mail_from, [payload.to], msg.as_string())

    # ----------------------------
    # Payload validation
    # ----------------------------
    @staticmethod
    def _parse_email_payload(payload: Dict[str, Any]) -> EmailPayload:
        to = payload.get("to")
        subject = payload.get("subject")
        html = payload.get("html")

        if not to or not subject or html is None:
            raise ValueError(f"Invalid EmailPayload: {json.dumps(payload, default=str)[:500]}")

        return EmailPayload(to=str(to), subject=str(subject), html=str(html))

    def get_worker_stats(self) -> Dict[str, Any]:
        return {"is_processing": self.is_processing, "schedule": "external (no cron)"}


In [10]:
from typing import Sequence, List
from datetime import datetime

class InMemoryAsyncTaskService:
    def __init__(self, tasks: Sequence[AsyncTask]) -> None:
        self._tasks = list(tasks)

    def get_pending_tasks_for_processing(self, limit: int) -> List[AsyncTask]:
        return self._tasks[:limit]

    def mark_as_processing(self, task_id: str) -> None:
        return

    def mark_as_completed(self, task_id: str) -> None:
        self._tasks = [t for t in self._tasks if t.task_id != task_id]

    def mark_as_failed(self, task_id: str, retry_count: int) -> None:
        self._tasks = [t for t in self._tasks if t.task_id != task_id]

    def schedule_retry(self, task_id: str, retry_count: int, next_retry_at: datetime) -> None:
        for i, t in enumerate(self._tasks):
            if t.task_id == task_id:
                self._tasks[i] = AsyncTask(
                    task_id=t.task_id,
                    payload=t.payload,
                    retry_count=retry_count,
                    max_retries=t.max_retries,
                )


In [13]:
if __name__ == "__main__":
    logging.basicConfig(level=logging.INFO)

    svc = InMemoryAsyncTaskService([
        AsyncTask(
            task_id="t1",
            payload={"to": "aliza.ashfaq@paysyslabs.com", "subject": "Test", "html": "<b>Hello</b>"},
            retry_count=0,
            max_retries=5,
        )
    ])

    worker = EmailWorkerService(svc)
    worker.process_email_queue(limit=10)

INFO:EmailWorkerService:Email Worker initialized - SMTP: smtp.gmail.com:587
INFO:EmailWorkerService:Processing 1 email tasks
INFO:EmailWorkerService:Email sent: t1 to aliza.ashfaq@paysyslabs.com - Test


# Alerts 

In [14]:
from pyspark.sql import SparkSession
from pyspark import SparkConf

In [ ]:
# Using SPARK_HOME from environment (already set in Docker)
# os.environ['SPARK_HOME'] = os.environ.get('SPARK_HOME', '/usr/local/lib/python3.10/dist-packages/pyspark')

# Specify the matching version JARs
jar_files = [
    "' + os.path.join(os.getcwd(), 'hudi-spark3.4-bundle_2.12-0.14.1.jar') + '",
    "' + os.path.join(os.getcwd(), 'spark-3.4.2-bin-hadoop3') + '/jars/hadoop-aws-3.3.4.jar",
    "' + os.path.join(os.getcwd(), 'spark-3.4.2-bin-hadoop3') + '/jars/aws-java-sdk-bundle-1.12.262.jar"
]

spark = (
    SparkSession.builder
    .appName("OzoneS3A-FIXED-v3.3.4")
    .enableHiveSupport()
    .master("local[*]")
    .config("spark.jars", ",".join(jar_files))
    .config("spark.driver.extraClassPath", ":".join(jar_files))
    .config("spark.executor.extraClassPath", ":".join(jar_files))
    .config("spark.hadoop.fs.s3a.endpoint", "http://localhost:9878")
    .config("spark.hadoop.fs.s3a.access.key", "admin")
    .config("spark.hadoop.fs.s3a.secret.key", "admin")
    .config("spark.hadoop.fs.s3a.path.style.access", "true")
    .config("spark.hadoop.fs.s3a.connection.ssl.enabled", "false")
    .config("spark.hadoop.fs.s3a.impl", "org.apache.hadoop.fs.s3a.S3AFileSystem")
    .config("spark.hadoop.fs.s3a.impl.disable.cache", "true")
    # Hudi configurations (removed problematic extensions)
    .config("spark.serializer", "org.apache.spark.serializer.KryoSerializer")
    .config("spark.kryo.registrator", "org.apache.spark.HoodieSparkKryoRegistrar")
    .config("spark.sql.extensions", "org.apache.spark.sql.hudi.HoodieSparkSessionExtension")
    .config("spark.sql.catalog.spark_catalog", "org.apache.spark.sql.hudi.catalog.HoodieCatalog")

    # Performance and optimization
    .config("spark.driver.memory", "4g")
    .config("spark.executor.memory", "4g")
    .config("spark.driver.maxResultSize", "2g")
    .config("spark.sql.shuffle.partitions", "200")
    .config("spark.default.parallelism", "200")
    .config("spark.sql.adaptive.enabled", "true")
    .config("spark.sql.adaptive.coalescePartitions.enabled", "true")
    # Time and legacy settings
    .config("spark.sql.legacy.timeParserPolicy", "LEGACY")
    .config("spark.sql.session.timeZone", "UTC")
    .getOrCreate()
)

spark.sparkContext.setLogLevel("WARN")
print(f"Spark Version: {spark.version}")


26/01/23 05:30:07 WARN NativeCodeLoader: Unable to load native-hadoop library for your platform... using builtin-java classes where applicable
Setting default log level to "WARN".
To adjust logging level use sc.setLogLevel(newLevel). For SparkR, use setLogLevel(newLevel).
26/01/23 05:30:10 WARN Utils: Service 'SparkUI' could not bind on port 4040. Attempting port 4041.


Spark Version: 3.4.2


In [ ]:
from datetime import datetime
import pyspark.sql.functions as F
from pyspark.sql.window import Window

WAREHOUSE_ROOT = os.environ.get("WAREHOUSE_ROOT", os.path.join(os.getcwd(), "Tazama_Hudi_warehouse"))

gold_alerts_path =  f"{WAREHOUSE_ROOT}/gold/alerts"
gold_alert_notifications_path = f"{WAREHOUSE_ROOT}/gold/alert_notifications" 

# Email routing 
EMAIL_TO_FRAUD = "arqam.wadiwala@paysyslabs.com"
EMAIL_TO_AML = "arqam.wadiwala@paysyslabs.com"
EMAIL_TO_DEFAULT = "arqam.wadiwala@paysyslabs.com"

# "High priority" definition (matches your data)
HIGH_PRIORITY_NORMS = ["URGENT", "BREACH"]
HIGH_PRIORITY_SCORE_THRESHOLD = 80.0  # adjust if needed

# Alert types requiring emails
ALERT_TYPES = ["AML", "FRAUD", "FRAUD_AND_AML"]

In [40]:
def latest_hudi_snapshot(df, key_col: str):
    w = Window.partitionBy(key_col).orderBy(F.col("_hoodie_commit_time").desc(), F.col("_hoodie_commit_seqno").desc())
    return df.withColumn("_rn", F.row_number().over(w)).filter(F.col("_rn")==1).drop("_rn")

alerts_gold = spark.read.format("hudi").load(gold_alerts_path)
alerts_gold_latest = latest_hudi_snapshot(alerts_gold, "alert_id")


In [41]:
from pyspark.sql.types import StructType, StructField, LongType, StringType, TimestampType

notif_schema = StructType([
    StructField("alert_id", LongType(), False),
    StructField("notified_at_ts", TimestampType(), True),
    StructField("notification_type", StringType(), True),  # e.g. EMAIL
    StructField("email_to", StringType(), True),
    StructField("email_subject", StringType(), True),
    StructField("email_status", StringType(), True),       # CREATED/SENT/FAILED
    StructField("created_at_ts", TimestampType(), True),
])

try:
    notified = spark.read.format("hudi").load(gold_alert_notifications_path)
    notified = latest_hudi_snapshot(notified, "alert_id").select("alert_id")
except Exception:
    # first run: no table yet
    notified = spark.createDataFrame([], notif_schema).select("alert_id")


In [42]:
# Candidates based on your gold schema
candidates = (
    alerts_gold_latest
    .filter(F.col("alert_type_norm").isNotNull())
    .filter(F.upper(F.col("alert_type_norm")).isin(ALERT_TYPES))
    .filter(
        (F.upper(F.col("priority_norm")).isin(HIGH_PRIORITY_NORMS)) |
        (F.col("priority_score").cast("double") >= F.lit(HIGH_PRIORITY_SCORE_THRESHOLD))
    )
)

# Remove already-notified alerts (idempotent)
new_candidates = candidates.join(notified, on="alert_id", how="left_anti")

print("New high-priority AML/Fraud alerts to notify:", new_candidates.count())


New high-priority AML/Fraud alerts to notify: 54


In [43]:
new_candidates.show()

+--------+-------------------+--------------------+------------------+----------------------+--------------------+-------+----------+-------------+--------------+---------------+-----------------------+-------+---------------+--------------------+--------------------+------------+--------------------+---------------+--------------------+---------+---------+------+--------------------+--------------+--------------------+----------------+----------------------+---------------+---------------+---------------+------------------+---------------+-----------+---------------+------------+-------------------+---------------------+--------------------------+---------------------+------------------+------------------------+-----------------+--------------------+--------------------+----------+
|alert_id|_hoodie_commit_time|_hoodie_commit_seqno|_hoodie_record_key|_hoodie_partition_path|   _hoodie_file_name|case_id| tenant_id|priority_norm|priority_score|alert_type_norm|prediction_outcome_norm| sou

In [44]:
def email_to_expr():
    return (
        F.when(F.upper(F.col("alert_type_norm")) == "AML", F.lit(EMAIL_TO_AML))
         .when(F.upper(F.col("alert_type_norm")) == "FRAUD", F.lit(EMAIL_TO_FRAUD))
         .when(F.upper(F.col("alert_type_norm")) == "FRAUD_AND_AML", F.lit(EMAIL_TO_DEFAULT))
         .otherwise(F.lit(EMAIL_TO_DEFAULT))
    )

new_emails = (
    new_candidates
    .withColumn("email_to", email_to_expr())
    .withColumn(
        "email_subject",
        F.concat(
            F.lit("["),
            F.coalesce(F.col("priority_norm"), F.lit("INFO")),
            F.lit("] "),
            F.lit("New "),
            F.coalesce(F.col("alert_type_norm"), F.lit("UNKNOWN")),
            F.lit(" Alert - "),
            F.col("alert_id").cast("string")
        )
    )
    .withColumn(
        "email_html",
        F.concat(
            F.lit("<html><body>"),
            F.lit("<h3>New High-Priority Alert</h3><ul>"),
            F.lit("<li><b>alert_id:</b> "), F.col("alert_id").cast("string"), F.lit("</li>"),
            F.lit("<li><b>tenant_id:</b> "), F.col("tenant_id"), F.lit("</li>"),
            F.lit("<li><b>alert_type:</b> "), F.col("alert_type_norm"), F.lit("</li>"),
            F.lit("<li><b>priority:</b> "), F.col("priority_norm"), F.lit("</li>"),
            F.lit("<li><b>priority_score:</b> "), F.col("priority_score").cast("string"), F.lit("</li>"),
            F.lit("<li><b>alert_status:</b> "), F.col("alert_status"), F.lit("</li>"),
            F.lit("<li><b>prediction_outcome:</b> "), F.col("prediction_outcome_norm"), F.lit("</li>"),
            F.lit("<li><b>event_ts:</b> "), F.col("event_ts").cast("string"), F.lit("</li>"),
            F.lit("<li><b>created_at_ts:</b> "), F.col("created_at_ts").cast("string"), F.lit("</li>"),
            F.lit("<li><b>typology_id:</b> "), F.col("typology_id"), F.lit("</li>"),
            F.lit("<li><b>top_rule_id:</b> "), F.col("top_rule_id"), F.lit("</li>"),
            F.lit("<li><b>top_rule_weight:</b> "), F.col("top_rule_weight").cast("string"), F.lit("</li>"),
            F.lit("<li><b>tx_amount:</b> "), F.col("tx_amount").cast("string"), F.lit(" "), F.col("tx_ccy"), F.lit("</li>"),
            F.lit("</ul></body></html>")
        )
    )
)


In [45]:
notif_rows = (
    new_emails
    .select(
        F.col("alert_id").cast("long"),
        F.current_timestamp().alias("notified_at_ts"),
        F.lit("EMAIL").alias("notification_type"),
        F.col("email_to"),
        F.col("email_subject"),
        F.lit("CREATED").alias("email_status"),
        F.current_timestamp().alias("created_at_ts"),
    )
)

notif_hudi_opts = {
    "hoodie.table.name": "alert_notifications",
    "hoodie.datasource.write.table.type": "COPY_ON_WRITE",
    "hoodie.datasource.write.operation": "upsert",
    "hoodie.datasource.write.recordkey.field": "alert_id",
    "hoodie.datasource.write.precombine.field": "notified_at_ts",
    "hoodie.datasource.write.keygenerator.class": "org.apache.hudi.keygen.NonpartitionedKeyGenerator",
    "hoodie.datasource.write.schema.evolution.enable": "true",
    "hoodie.datasource.write.reconcile.schema": "true",
    "hoodie.schema.on.read.enable": "true",
    "hoodie.metadata.enable": "false",
}

(
    notif_rows.write.format("hudi")
    .options(**notif_hudi_opts)
    .mode("append")
    .save(gold_alert_notifications_path)
)

print("Notification log updated:", notif_rows.count(), "records")


26/01/23 05:57:25 WARN HoodieSparkSqlWriterInternal: Closing write client       


Notification log updated: 54 records


In [46]:
spark.read.format("hudi") \
    .load(gold_alert_notifications_path) \
    .select(
        "alert_id",
        "email_to",
        "email_subject",
        "email_status",
        "notified_at_ts"
    ) \
    .orderBy(F.desc("notified_at_ts")) \
    .show(20, truncate=False)

+--------+-----------------------------+----------------------------------------+------------+--------------------------+
|alert_id|email_to                     |email_subject                           |email_status|notified_at_ts            |
+--------+-----------------------------+----------------------------------------+------------+--------------------------+
|375     |arqam.wadiwala@paysyslabs.com|[CRITICAL] New FRAUD_AND_AML Alert - 375|CREATED     |2026-01-23 10:57:14.189167|
|448     |arqam.wadiwala@paysyslabs.com|[BREACH] New FRAUD_AND_AML Alert - 448  |CREATED     |2026-01-23 10:57:14.189167|
|380     |arqam.wadiwala@paysyslabs.com|[BREACH] New FRAUD_AND_AML Alert - 380  |CREATED     |2026-01-23 10:57:14.189167|
|186     |arqam.wadiwala@paysyslabs.com|[BREACH] New AML Alert - 186            |CREATED     |2026-01-23 10:57:14.189167|
|175     |arqam.wadiwala@paysyslabs.com|[BREACH] New FRAUD_AND_AML Alert - 175  |CREATED     |2026-01-23 10:57:14.189167|
|85      |arqam.wadiwala

In [49]:
notif_rows.show()

+--------+--------------------+-----------------+--------------------+--------------------+------------+--------------------+
|alert_id|      notified_at_ts|notification_type|            email_to|       email_subject|email_status|       created_at_ts|
+--------+--------------------+-----------------+--------------------+--------------------+------------+--------------------+
|      85|2026-01-23 10:59:...|            EMAIL|arqam.wadiwala@pa...|[URGENT] New FRAU...|     CREATED|2026-01-23 10:59:...|
|      86|2026-01-23 10:59:...|            EMAIL|arqam.wadiwala@pa...|[URGENT] New FRAU...|     CREATED|2026-01-23 10:59:...|
|      87|2026-01-23 10:59:...|            EMAIL|arqam.wadiwala@pa...|[URGENT] New AML ...|     CREATED|2026-01-23 10:59:...|
|     166|2026-01-23 10:59:...|            EMAIL|arqam.wadiwala@pa...|[BREACH] New AML ...|     CREATED|2026-01-23 10:59:...|
|     168|2026-01-23 10:59:...|            EMAIL|arqam.wadiwala@pa...|[URGENT] New FRAU...|     CREATED|2026-01-23 10:

In [50]:
notif_df = (
    spark.read.format("hudi")
    .load(gold_alert_notifications_path)
    .filter(F.col("email_status") == "CREATED")
)


In [55]:
gold_alerts = spark.read.format("hudi").load(gold_alerts_path)

In [56]:
email_df = (
    notif_df
    .join(
        gold_alerts.select(
            "alert_id",
            "alert_type_norm",
            "priority_score",
            "tenant_id",
            "event_ts"
        ),
        on="alert_id",
        how="left"
    )
)

In [57]:
def build_email_html(row):
    return f"""
    <html>
      <body style="font-family: Arial, sans-serif;">
        <h2 style="color:#c0392b;">🚨 High Priority Alert</h2>

        <table cellpadding="6" cellspacing="0" border="0">
          <tr><td><b>Alert ID</b></td><td>{row.alert_id}</td></tr>
          <tr><td><b>Alert Type</b></td><td>{row.alert_type_norm or "UNKNOWN"}</td></tr>
          <tr><td><b>Priority Score</b></td><td>{row.priority_score or "N/A"}</td></tr>
          <tr><td><b>Tenant</b></td><td>{row.tenant_id or "N/A"}</td></tr>
          <tr><td><b>Event Time</b></td><td>{row.event_ts or "N/A"}</td></tr>
        </table>

        <br/>
        <p>Please review this alert immediately.</p>
      </body>
    </html>
    """


In [58]:
tasks = []

for r in email_df.collect():
    tasks.append(
        AsyncTask(
            task_id=f"email-{r.alert_id}",
            payload={
                "to": r.email_to,
                "subject": r.email_subject,
                "html": build_email_html(r),
            },
            retry_count=0,
            max_retries=5
        )
    )


In [61]:
service = EmailWorkerService(
    async_task_service=InMemoryAsyncTaskService(tasks)
)

service.run_once()

INFO:EmailWorkerService:Email Worker initialized - SMTP: smtp.gmail.com:587
INFO:EmailWorkerService:Processing 10 email tasks
INFO:EmailWorkerService:Email sent: email-375 to arqam.wadiwala@paysyslabs.com - [CRITICAL] New FRAUD_AND_AML Alert - 375
INFO:EmailWorkerService:Email sent: email-448 to arqam.wadiwala@paysyslabs.com - [BREACH] New FRAUD_AND_AML Alert - 448
INFO:EmailWorkerService:Email sent: email-380 to arqam.wadiwala@paysyslabs.com - [BREACH] New FRAUD_AND_AML Alert - 380
INFO:EmailWorkerService:Email sent: email-186 to arqam.wadiwala@paysyslabs.com - [BREACH] New AML Alert - 186
INFO:EmailWorkerService:Email sent: email-175 to arqam.wadiwala@paysyslabs.com - [BREACH] New FRAUD_AND_AML Alert - 175
INFO:EmailWorkerService:Email sent: email-85 to arqam.wadiwala@paysyslabs.com - [URGENT] New FRAUD Alert - 85


KeyboardInterrupt: 